# Air Quality Prediction and Pollution Monitoring System

Run this notebook from the project folder (the folder that contains `data/air_quality_50.csv`). Run cells top to bottom.

### 1. Import the Required Libraries

In [ ]:
%matplotlib inline
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import f1_score, confusion_matrix

### 2. Load the Dataset

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

df = pd.read_csv("data/air_quality_50.csv")
print("Dataset loaded successfully.")
print(df.head())

### 3. Understand the Dataset

In [ ]:
print("Dataset Shape:", df.shape)
print("\nColumn Names:")
print(df.columns)
print("\nDataset Information:")
df.info()
print("\nStatistical Summary:")
print(df.describe())

### 4. Check Missing Values and Clean the Data

In [ ]:
print("Missing Values:")
print(df.isnull().sum())
df = df.dropna()
df = df.drop_duplicates()
print("Shape after cleaning:", df.shape)

### 5. Basic Data Visualization

In [ ]:
class_order = ["Low", "Moderate", "High"]
class_colors = {"Low": "green", "Moderate": "orange", "High": "red"}

plt.figure(figsize=(8, 5))
counts = df["Pollution_Level"].value_counts().reindex(class_order)
counts.plot(kind="bar", color=[class_colors[c] for c in class_order])
plt.title("Pollution Level Distribution")
plt.xlabel("Pollution Level")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.show()

plt.figure(figsize=(8, 5))
for level in class_order:
    part = df[df["Pollution_Level"] == level]
    plt.scatter(part["PM2_5"], part["PM10"], alpha=0.7,
                color=class_colors[level], label=level)
plt.title("PM2.5 vs PM10 by Pollution Level")
plt.xlabel("PM2.5 (µg/m³)")
plt.ylabel("PM10 (µg/m³)")
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
groups = [df[df["Pollution_Level"] == c]["PM2_5"] for c in class_order]
plt.boxplot(groups)
plt.xticks([1, 2, 3], class_order)
plt.title("PM2.5 Concentration by Pollution Level")
plt.xlabel("Pollution Level")
plt.ylabel("PM2.5 (µg/m³)")
plt.show()

### 6. Select Input Features and Target

In [ ]:
features = ["PM2_5", "PM10", "NO2", "SO2", "CO", "O3", "Temperature", "Humidity"]
class_map = {"Low": 0, "Moderate": 1, "High": 2}

X = df[features]
y = df["Pollution_Level"].map(class_map)
print("Input shape:", X.shape)
print("Target shape:", y.shape)

### 7. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

### 8. Logistic Regression Model

In [ ]:
log_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced",
                                      random_state=42))
])
log_model.fit(X_train, y_train)
log_predictions = log_model.predict(X_test)
print("Logistic Regression trained successfully.")

### 9. Evaluate Logistic Regression

In [ ]:
log_acc = accuracy_score(y_test, log_predictions)
log_prec = precision_score(y_test, log_predictions, average="macro", zero_division=0)
log_rec = recall_score(y_test, log_predictions, average="macro", zero_division=0)
log_f1 = f1_score(y_test, log_predictions, average="macro", zero_division=0)
print("Logistic Regression Results")
print("Accuracy :", round(log_acc, 4))
print("Precision:", round(log_prec, 4))
print("Recall   :", round(log_rec, 4))
print("F1-Score :", round(log_f1, 4))

### 10. Random Forest Model

In [ ]:
rf_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                          random_state=42))
])
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)
print("Random Forest trained successfully.")

### 11. Evaluate Random Forest

In [ ]:
rf_acc = accuracy_score(y_test, rf_predictions)
rf_prec = precision_score(y_test, rf_predictions, average="macro", zero_division=0)
rf_rec = recall_score(y_test, rf_predictions, average="macro", zero_division=0)
rf_f1 = f1_score(y_test, rf_predictions, average="macro", zero_division=0)
print("Random Forest Results")
print("Accuracy :", round(rf_acc, 4))
print("Precision:", round(rf_prec, 4))
print("Recall   :", round(rf_rec, 4))
print("F1-Score :", round(rf_f1, 4))

### 12. Compare Models

In [ ]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [log_acc, rf_acc],
    "Precision": [log_prec, rf_prec],
    "Recall": [log_rec, rf_rec],
    "F1-Score": [log_f1, rf_f1]
})
print(results)

cm = confusion_matrix(y_test, rf_predictions, labels=[0, 1, 2])
print("\nConfusion Matrix (Low, Moderate, High):")
print(cm)

plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.xticks([0, 1, 2], class_order)
plt.yticks([0, 1, 2], class_order)
for i in range(3):
    for j in range(3):
        txt_color = "white" if cm[i, j] > cm.max() / 2 else "black"
        plt.text(j, i, cm[i, j], ha="center", va="center", color=txt_color)
plt.title("Confusion Matrix (Random Forest)")
plt.xlabel("Predicted Level")
plt.ylabel("Actual Level")
plt.show()

### 13. Feature Importance

In [ ]:
rf_classifier = rf_model.named_steps["classifier"]
importances = pd.Series(rf_classifier.feature_importances_,
                        index=features).sort_values()
plt.figure(figsize=(8, 5))
importances.plot(kind="barh")
plt.title("Feature Importance (Random Forest)")
plt.show()

### 14. Predict Pollution Level for New Data

In [ ]:
new_data = pd.DataFrame({
    "PM2_5": [120.0],
    "PM10": [210.0],
    "NO2": [60.0],
    "SO2": [30.0],
    "CO": [1.5],
    "O3": [90.0],
    "Temperature": [28.0],
    "Humidity": [55.0]
})
pred_index = int(rf_model.predict(new_data)[0])
pred_proba = rf_model.predict_proba(new_data)[0]
print("Predicted Pollution Level:", class_order[pred_index])
for name, p in zip(class_order, pred_proba):
    print(name, "probability:", round(float(p), 4))

### 15. Save the Trained Model

In [ ]:
final_model = clone(rf_model)
final_model.fit(X, y)

os.makedirs("models", exist_ok=True)
joblib.dump({
    "pipeline": final_model,
    "feature_columns": features,
    "class_names": class_order
}, "models/air_quality_model.joblib")
print("Model saved as models/air_quality_model.joblib")